# ResNet18 — Histopathologic Cancer Detection

Colab notebook version based on the provided ResNet18 experiment.

- Pretrained ResNet18
- 70/30 train-validation split
- Fine-tuning of `layer3`, `layer4`, and `fc`
- SGD + momentum
- CyclicLR scheduler
- 10 epochs
- Local Colab checkpoint saving
- Confusion matrix
- ROC/AUC curve

This version uses KaggleHub for the competition dataset and avoids Google Drive mounting.


## 1. Imports

In [ ]:
import os
from pathlib import Path

import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms

## 2. Download Dataset with KaggleHub

In [ ]:
import kagglehub

kagglehub.login()

path = kagglehub.competition_download(
    "histopathologic-cancer-detection"
)

DATA_DIR = Path(path)

print("Dataset directory:")
print(DATA_DIR)

print("\nFiles inside dataset directory:")
for item in DATA_DIR.iterdir():
    print(item.name)

## 3. Dataset Paths

In [ ]:
csv_path = DATA_DIR / "train_labels.csv"
root_dir = DATA_DIR / "train"

print("CSV path:", csv_path)
print("Train folder:", root_dir)

print("CSV exists:", csv_path.exists())
print("Train folder exists:", root_dir.exists())

### If `train` does not exist

Some Kaggle downloads may contain `train.zip` instead of an already extracted `train/` folder.
Run the next cell only if `Train folder exists: False`.


In [ ]:
import zipfile

train_zip = DATA_DIR / "train.zip"

if not root_dir.exists() and train_zip.exists():

    print("Extracting train.zip...")

    root_dir = Path("/content/train")

    with zipfile.ZipFile(train_zip, "r") as zip_ref:
        zip_ref.extractall(root_dir)

    print("Extraction complete!")

print("CSV exists:", csv_path.exists())
print("Train folder exists:", root_dir.exists())

## 4. Custom TIFF Dataset

In [ ]:
def tiff_loader(path: str) -> Image.Image:
    with open(path, 'rb') as f:
        img = Image.open(f)
        return img.convert('RGB')


class CustomTiffDataset(Dataset):

    def __init__(
        self,
        csv_file,
        root_dir,
        transform=None,
        loader=tiff_loader
    ):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.loader = loader

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):

        img_id = self.annotations.iloc[idx, 0]

        img_path = os.path.join(
            self.root_dir,
            f"{img_id}.tif"
        )

        image = self.loader(img_path)

        label = int(
            self.annotations.iloc[idx, 1]
        )

        if self.transform:
            image = self.transform(image)

        return image, label

## 5. Image Transforms

In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

## 6. Create Dataset and Split 70/30

In [ ]:
full_dataset = CustomTiffDataset(
    csv_file=csv_path,
    root_dir=root_dir,
    transform=data_transforms
)

dataset_length = len(full_dataset)

train_length = int(
    0.7 * dataset_length
)

val_length = (
    dataset_length - train_length
)

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_length, val_length]
)

print("Total images:", dataset_length)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

## 7. DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

## 8. Prepare ResNet18

In [ ]:
model = models.resnet18(
    pretrained=True
)

for name, param in model.named_parameters():

    if (
        "layer3" in name
        or "layer4" in name
        or "fc" in name
    ):
        param.requires_grad = True

    else:
        param.requires_grad = False


num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    2
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 9. Loss, Optimizer, and Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=0.0005,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.CyclicLR(
    optimizer,
    base_lr=1e-6,
    max_lr=1e-3,
    step_size_up=200,
    mode='triangular2'
)

## 10. Checkpoint Directory

In [ ]:
checkpoint_dir = "/content/checkpoints_resnet18"

os.makedirs(
    checkpoint_dir,
    exist_ok=True
)

print(
    "Checkpoint directory:",
    checkpoint_dir
)

## 11. Training and Validation

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    running_train_loss = 0.0
    running_train_corrects = 0
    train_total = 0

    for inputs, labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        scheduler.step()

        running_train_loss += (
            loss.item()
            * inputs.size(0)
        )

        _, preds = torch.max(
            outputs,
            1
        )

        running_train_corrects += torch.sum(
            preds == labels.data
        ).item()

        train_total += inputs.size(0)


    epoch_train_loss = (
        running_train_loss
        / train_total
    )

    epoch_train_acc = (
        running_train_corrects
        / train_total
    )


    model.eval()

    running_val_loss = 0.0
    running_val_corrects = 0
    val_total = 0

    with torch.no_grad():

        for inputs, labels in val_loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            loss = criterion(
                outputs,
                labels
            )

            running_val_loss += (
                loss.item()
                * inputs.size(0)
            )

            _, preds = torch.max(
                outputs,
                1
            )

            running_val_corrects += torch.sum(
                preds == labels.data
            ).item()

            val_total += inputs.size(0)


    epoch_val_loss = (
        running_val_loss
        / val_total
    )

    epoch_val_acc = (
        running_val_corrects
        / val_total
    )


    print(
        f"Epoch {epoch+1}/{num_epochs}"
    )

    print(
        f"  Training   - Loss: {epoch_train_loss:.4f}, "
        f"Accuracy: {epoch_train_acc:.4f}"
    )

    print(
        f"  Validation - Loss: {epoch_val_loss:.4f}, "
        f"Accuracy: {epoch_val_acc:.4f}"
    )


    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"resnet18_epoch{epoch+1}.pth"
    )

    torch.save(
        model.state_dict(),
        checkpoint_path
    )

    print(
        f"Checkpoint saved to {checkpoint_path}"
    )

## 12. Evaluation Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc
)

## 13. Confusion Matrix

In [ ]:
def plot_confusion_matrix(
    model,
    dataloader,
    device
):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for inputs, labels in dataloader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            _, preds = torch.max(
                outputs,
                1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )


    cm = confusion_matrix(
        all_labels,
        all_preds
    )


    plt.figure(
        figsize=(6, 5)
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[
            "Class 0",
            "Class 1"
        ],
        yticklabels=[
            "Class 0",
            "Class 1"
        ]
    )

    plt.xlabel(
        "Predicted Labels"
    )

    plt.ylabel(
        "True Labels"
    )

    plt.title(
        "Confusion Matrix"
    )

    plt.show()


plot_confusion_matrix(
    model,
    val_loader,
    device
)

## 14. ROC Curve and AUC

In [ ]:
def plot_roc_curve(
    model,
    dataloader,
    device
):

    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():

        for inputs, labels in dataloader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            probs = torch.nn.functional.softmax(
                outputs,
                dim=1
            )[:, 1]

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )


    fpr, tpr, _ = roc_curve(
        all_labels,
        all_probs
    )

    roc_auc = auc(
        fpr,
        tpr
    )


    plt.figure(
        figsize=(6, 5)
    )

    plt.plot(
        fpr,
        tpr,
        color='blue',
        lw=2,
        label=f"ROC curve (AUC = {roc_auc:.2f})"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        color='gray',
        linestyle="--"
    )

    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.title(
        "ROC Curve"
    )

    plt.legend(
        loc="lower right"
    )

    plt.show()


plot_roc_curve(
    model,
    val_loader,
    device
)